In [ ]:
import numpy as np
from PIL import Image
import torchvision.transforms as T
import torch
from torch import nn
from torchvision.models import vgg19
from torch.nn.functional import mse_loss
from torch import optim
from torchvision.utils import save_image


def norm_images(image_arr: np.typing.NDArray[float], size: int = 512) -> torch.Tensor:
    transform = T.Compose([
        T.Resize(size),
        T.ToTensor(),
        T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    img = transform(image_arr)

    return img.unsqueeze(0)

class VGGExtractor(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.features = ['0', '5', '10', '19', '28']
        self.vgg = vgg19(weights=True).features

    def forward(self, x):
        features = []
        for name, layer in self.vgg._modules.items():
            x = layer(x)
            if name in self.features:
                features.append(x)

        return features

def gram_matrix(data) -> torch.Tensor:
    _, c, h, w = data.shape
    data_flatten = data.reshape((c, h * w))
    return data_flatten @ data_flatten.T

def total_loss(alfa, beta, content, style, generated):
    content_loss = 0
    style_loss = 0
    for content_feature, style_feature, generated_feature in zip(content, style, generated):
        _, c, h, w = style_feature.shape
        G, S = gram_matrix(style_feature), gram_matrix(generated_feature)
        style_loss += mse_loss(G, S) / (4 * (c ** 2) * (h * w) ** 2)
        content_loss += mse_loss(content_feature, generated_feature)

    return alfa * content_loss + beta * style_loss

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

content = Image.open("Locha.jpg")
style = Image.open("style.jpg")

ready_content = norm_images(content).to(device=device)
ready_generated = ready_content.clone().requires_grad_(True).to(device=device)
ready_style = norm_images(style).to(device=device)

In [ ]:
optimizer = optim.Adam([ready_generated], lr=0.01)

steps, alfa, beta = 100, 0.01, 200000

vgg = VGGExtractor().to(device=device).eval()
for step in range(steps):
    generated_feature = vgg(ready_generated)
    content_feature = vgg(ready_content)
    style_feature = vgg(ready_style)

    loss = total_loss(alfa, beta, content_feature, style_feature, generated_feature)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f"Loss = {loss}")

In [ ]:
def save(generated, i):
    denormalization = T.Normalize((-2.12, -2.04, -1.80), (4.37, 4.46, 4.44))
    img = generated.clone().squeeze()
    img = denormalization(img).clamp(0, 1)
    save_image(img, f'result_{i}.png')

save(ready_generated, 0)